# Chapter 12: the price ladder

The State S walk for each Kestrel SKU, the bands against a synthetic popular price of INR 190, contribution
a case and the trade up arithmetic. Mark pairs less than INR 20 apart. Every figure is synthetic.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from calculator import inr, r2
import math
from calculator import walk, load_rates, reverse_dmc

In [ ]:
rates = load_rates()
sku = pd.read_csv('../data/skus.csv').set_index('sku_id')
ANCHOR = 190
def band(x):
    return 'economy' if x < 0.9 else 'popular' if x <= 1.1 else 'premium' if x <= 1.5 else 'super premium'
rows = []
for sid in ['SKU1', 'SKU4', 'SKU5', 'SKU2']:
    s = sku.loc[sid]
    w = walk(s.dmc_case, int(s.units), s.litres_per_unit, s.abv, rates=rates)
    beer_pack_freight = r2(s.hl_case * 2880) + r2(s.units * s.unit_cost + s.carton_cost) + r2(s.hl_case * 630)
    rows.append(dict(sku=f'{s.beer}, {s.pack}', dmc=s.dmc_case, before_rounding=w.pre_round_bottle, mrp=w.mrp_bottle,
                     vs_anchor=round(w.mrp_bottle / ANCHOR, 2), band=band(w.mrp_bottle / ANCHOR),
                     mrp_litre=round(w.mrp_bottle / s.litres_per_unit, 2), contribution=r2(s.dmc_case - beer_pack_freight)))
ladder = pd.DataFrame(rows).set_index('sku')
ladder

In [ ]:
mrps = ladder.mrp.iloc[:3].sort_values()
close = [(a, b) for a, b in zip(mrps.index, mrps.index[1:]) if mrps[b] - mrps[a] < 20]
print('pairs less than INR 20 apart:', close)
shelf_per_rupee = round((1 + 1.10) * 1.08 * 1.15, 2)
print('a rupee of DMC puts INR', shelf_per_rupee, 'on the shelf')
mild = walk(982.80, abv=5.0, rates=rates)
mild_ceiling = r2(reverse_dmc(220, share=0.80))
print('brewed at 5.0 percent: INR', mild.pre_round_bottle, 'a bottle, MRP', mild.mrp_bottle, '; DMC ceiling at 220:', inr(mild_ceiling))

In [ ]:
trade_up = r2(ladder.contribution.iloc[2] - ladder.contribution.iloc[0])
can_loss = r2(463.02 - 397.52)
print('lager to pale ale: +', trade_up, 'a case; 100 cases +', inr(100 * trade_up))
print('lager to a can case: -', can_loss, 'a case')
print('registration repaid by', math.ceil(25000 / ladder.contribution.iloc[2]), 'new pale ale cases or', math.ceil(25000 / trade_up), 'switched from lager')
# Elasticity from one episode, midpoints (invented teaching numbers)
dv = (3700 - 4000) / ((3700 + 4000) / 2); dp = (240 - 220) / ((240 + 220) / 2)
elasticity = round(dv * 100, 1) / round(dp * 100, 1)
print(f'volume {dv:.1%}, price {dp:.1%}, elasticity {elasticity:.2f}')